# Assignment 1

Deadline: 19.03.2026, 12:00 CET

<Add your name, student-id and emal address>

In [48]:
# Import standard libraries
import os
import sys
import timeit # To compute runtimes
from typing import Optional

# Import third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#/Users/oscar/Documents/QPWP/qpmwp-course/
# Import local modules
project_root = '/Users/oscar/Documents/QPWP/qpmwp-course/'  # Change this path if needed
src_path = os.path.join(project_root, 'src')
sys.path.append(project_root)
sys.path.append(src_path)
from estimation.covariance import Covariance, is_pos_def, make_pos_def
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.optimization import Optimization, Objective
from optimization.optimization_data import OptimizationData
from optimization.quadratic_program import QuadraticProgram, USABLE_SOLVERS
from helper_functions import simulate_correlated_gbm


## 1. Solver Horse Race

### 1.a)
(3 points)

Generate a synthetic dataset of dimension TxN, T=1000, N=50, and compute a vector of expected returns, q, and a covariance matrix, P, using classes ExpectedReturn and Covariance respectively.

In [52]:

# Set the dimensions
T = 1000       # Number of time steps
N = 50         # Number of assets
rnd_seed = 42  # Random seed for reproducibility

# Set random seed for reproducibility
np.random.seed(rnd_seed)

# Generate a random mean vector and covariance matrix.
# Make sure the covariance matrix is positive definite.

mu = np.random.rand(N) * 0.1

# positive by A'T * A
A = np.random.randn(N, N)

#scale = 0.001
sigma = A.T @ A #* scale

# Generate correlated geometric Brownian motion paths and compute discrete returns
prices = simulate_correlated_gbm(mu=mu, sigma=sigma, T=T, random_seed=None)
returns = prices.pct_change().dropna()

# Compute the vector of expected returns from df using class ExpectedReturn

expected_return = ExpectedReturn(method='geometric', scalefactor=1)
q = expected_return.estimate(X=returns, inplace=False)
expected_return.estimate(X=returns, inplace=True)
# Compute the covariance matrix from df using class Covariance
cov = Covariance(method='pearson')
P= cov.estimate(X=returns, inplace=False)
cov.estimate(X=returns, inplace=True)


# Display the results
print("Vector of expected returns (q):")
print(q)

print("\nCovariance matrix (P):")
print(P)

Vector of expected returns (q):
Asset_1    -0.024736
Asset_2    -0.036649
Asset_3    -0.024365
Asset_4    -0.017586
Asset_5    -0.020907
Asset_6    -0.022523
Asset_7    -0.019060
Asset_8    -0.016905
Asset_9    -0.020322
Asset_10   -0.018334
Asset_11   -0.012135
Asset_12   -0.029043
Asset_13   -0.023281
Asset_14   -0.017328
Asset_15   -0.033533
Asset_16   -0.033561
Asset_17   -0.024844
Asset_18   -0.028827
Asset_19   -0.028541
Asset_20   -0.027750
Asset_21   -0.023985
Asset_22   -0.028527
Asset_23   -0.034756
Asset_24   -0.044031
Asset_25   -0.015696
Asset_26   -0.022522
Asset_27   -0.015474
Asset_28   -0.027933
Asset_29   -0.031269
Asset_30   -0.020568
Asset_31   -0.021402
Asset_32   -0.025972
Asset_33   -0.018527
Asset_34   -0.022016
Asset_35   -0.033596
Asset_36   -0.024997
Asset_37   -0.020771
Asset_38   -0.011850
Asset_39   -0.021741
Asset_40   -0.038190
Asset_41   -0.013659
Asset_42   -0.043790
Asset_43   -0.018347
Asset_44   -0.023933
Asset_45   -0.022230
Asset_46   -0.029551
As

### 1.b)
(3 points)

Instantiate a constraints object by injecting column names of the return series created in 1.a) as ids and add:
- a budget constaint (i.e., asset weights have to sum to one)
- lower bounds of 0.0 for all assets
- upper bounds of 0.2 for all assets
- group contraints such that the sum of the weights of the first 15 assets is <= 0.3, the sum of assets 16 to 45 is <= 0.4 and the sum of assets 41 to 50 is <= 0.5

In [25]:
# Instantiate the Constraints class
constraints = Constraints(ids = returns.columns.tolist())

# Add budget constraint
constraints.add_budget(rhs=1, sense='=')

# Add box constraints (i.e., lower and upper bounds)
constraints.add_box(lower=0, upper=0.2)

# Add linear constraints
G = pd.DataFrame(np.zeros((3, N)), columns=constraints.ids)
G.iloc[0, 0:15] = 1   # Group 1
G.iloc[1, 15:45] = 1  # Group 2
G.iloc[2, 40:50] = 1  # Group 3
h = pd.Series([0.3, 0.4, 0.5])

constraints.add_linear(G=G, sense='<=', rhs=h)

# Display some columns of the G matrix to verify that the constraints have been set up correctly
constraints.linear['G'][['Asset_1', 'Asset_15', 'Asset_16', 'Asset_40', 'Asset_41', 'Asset_50']]

,Asset_1,Asset_15,Asset_16,Asset_40,Asset_41,Asset_50
0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,1.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,1.0,1.0


### 1.c) 
(4 points)

Solve a Mean-Variance optimization problem (using coefficients P and q in the objective function) which satisfies the above defined constraints.
Repeat the task for all open-source solvers in qpsolvers that you could install and compare the results in terms of:

- runtime
- accuracy: value of the primal problem.
- reliability: are all constraints fulfilled? Extract primal residuals, dual residuals and duality gap.

Generate a DataFrame with the solvers as column names and the following row index: 'solution_found': bool, 'objective': float, 'primal_residual': float, 'dual_residual': float, 'duality_gap': float, 'runtime': float.

Put NA's for solvers where the optimization failed for some reason.




In [ ]:
# Extract the constraints in the format required by the solver
GhAb = constraints.to_GhAb()

# Define a dictionary to store the results in case a solver fails
result_on_fail =  {
    'solution_found': False,
    'objective_builtin': np.nan,
    'primal_residual': np.nan,
    'dual_residual' : np.nan,
    'duality_gap': np.nan,
    'runtime': np.nan,
}

# Loop over solvers, instantiate the quadratic program, solve it and store the results in a dictionary.
results_dic = {}
solvers = solvers_to_test = list(USABLE_SOLVERS)
for i in solvers:
    risk_aversion = 3

    qp = QuadraticProgram(
        P = P,
        q = q.vector.to_numpy() * -1,
        G = GhAb['G'],
        h = GhAb['h'],
        A = GhAb['A'],
        b = GhAb['b'],
        lb = constraints.box['lower'].to_numpy(),
        ub = constraints.box['upper'].to_numpy(),
        solver = i,
    )

    qp.problem_data

    qp.is_feasible()

    qp.solve()
    qp.objective_value()

    results_dic[i] = {'solution': qp.results.get('solution')}
    '''solution

    

    solution.found
    solution.primal_residual()
    solution.dual_residual()
    solution.duality_gap()[0]'''
df_horse_race = pd.DataFrame(results_dic)
print(df_horse_race)

TypeError: a bytes-like object is required, not 'DataFrame'

In [81]:


# 2. Extract constraints
GhAb = constraints.to_GhAb()

# 3. Define failure dictionary
result_on_fail =  {
    'solution_found': False,
    'objective': np.nan,
    'primal_residual': np.nan,
    'dual_residual' : np.nan,
    'duality_gap': np.nan,
    'runtime': np.nan,
}

results_dic = {}
solvers = list(USABLE_SOLVERS)

# 4. Helper function to safely convert to numpy arrays, ignoring None types
def safe_np(item):
    if item is None:
        return None
    if hasattr(item, 'to_numpy'):
        return item.to_numpy()
    return np.asarray(item)

for i in solvers:
    try:
        qp = QuadraticProgram(
            # Now cov.matrix and expected_return.vector are guaranteed to exist!
            P = cov.matrix.to_numpy(), 
            q = expected_return.vector.to_numpy().flatten() * -1, 
            G = safe_np(GhAb.get('G')),
            h = safe_np(GhAb.get('h')),
            A = safe_np(GhAb.get('A')),
            b = safe_np(GhAb.get('b')),
            lb = safe_np(constraints.box.get('lower')),
            ub = safe_np(constraints.box.get('upper')),
            solver = i,
        )

        start_time = time.time()
        qp.solve()
        end_time = time.time()
        
        solution = qp.results.get('solution')
        
        if solution is not None and solution.found:
            d_gap = solution.duality_gap()
            if isinstance(d_gap, (list, tuple, np.ndarray)):
                d_gap = d_gap[0] if len(d_gap) > 0 else np.nan

            results_dic[i] = {
                'solution_found': solution.found,
                'objective': qp.objective_value(),
                'primal_residual': solution.primal_residual(),
                'dual_residual': solution.dual_residual(),
                'duality_gap': d_gap,
                'runtime': end_time - start_time
            }
        else:
            results_dic[i] = result_on_fail
            
    except Exception as e:
        print(f"Solver '{i}' crashed with error: {e}")
        results_dic[i] = result_on_fail

# Generate DataFrame
df_horse_race = pd.DataFrame(results_dic)
display(df_horse_race)

Solver 'daqp' crashed with error: buffer source array is read-only


/Users/oscar/Documents/QPWP/qpmwp-course/.venv/lib/python3.12/site-packages/qpsolvers/solvers/quadprog_.py:120: UserWarning: quadprog raised a ValueError: buffer source array is read-only
  warnings.warn(f"quadprog raised a ValueError: {error_message}")


,daqp,cvxopt,quadprog,qpalm,osqp
solution_found,False,True,False,True,True
objective,NaN,0.016296,NaN,0.016298,0.016295
primal_residual,NaN,0.0,NaN,0.000013,0.000006
dual_residual,NaN,0.0,NaN,0.0,0.0
duality_gap,NaN,0.0,NaN,0.000002,0.000001
runtime,NaN,0.012063,NaN,0.007449,0.006413


Print and visualize the results

In [5]:
# <your code here>

## 2. Analytical Solution to Minimum-Variance Problem

(5 points)

- Create a `MinVariance` class that follows the structure of the `MeanVariance` class.
- Implement the `solve` method in `MinVariance` such that if `solver_name = 'analytical'`, the analytical solution is computed and stored within the object (if such a solution exists). If not, call the `solve` method from the parent class.
- Create a `Constraints` object by injecting the same ids as in part 1.b) and add a budget constraint.
- Instantiate a `MinVariance` object by setting `solver_name = 'analytical'` and passing instances of `Constraints` and `Covariance` as arguments.
- Create an `OptimizationData` object that contains an element `return_series`, which consists of the synthetic data generated in part 1.a).
- Solve the optimization problem using the created `MinVariance` object and compare the results to those obtained in part 1.c).


In [79]:
# Define class MinVariance
class MinVariance(Optimization):

    def __init__(self,
                 constraints: Constraints,
                 covariance: Optional[Covariance] = None,
                 **kwargs):
        super().__init__(
            constraints=constraints,
            **kwargs
        )
        self.covariance = Covariance() if covariance is None else covariance

    def set_objective(self, optimization_data: OptimizationData) -> None:
        X = optimization_data['return_series']
        self.covmat = self.covariance.estimate(X=X, inplace=False)
        mu = np.zeros(X.shape[1])
        self.objective = Objective(
            q = mu ,
            P = self.covmat * 2,
        )
        return None

    def solve(self) -> None:
        if self.params.get('solver_name') == 'analytical':
            cov = self.covmat
            if is_pos_def(cov): pass
            else: make_pos_def(cov)
            inv_cov = np.linalg.inv(cov)
            one_vec = np.ones(len(cov))

            num = inv_cov @ one_vec
            den = one_vec @ inv_cov @ one_vec
            weights = num / den
            
            self.results.update({
            'weights': weights,
            'status': 'analytical',
        })
            return None
        else:
            return super().solve()

opt_data = {'return_series': returns}
# Create a constraints object with just a budget constraint
constraints.add_budget(rhs=1, sense='=')

# Instantiate the MinVariance class
min_var = MinVariance(
    constraints= constraints,
    covariance=cov,
    solver_name='analytical'
)

# Prepare the optimization data and prepare the optimization problem
min_var.set_objective(optimization_data=opt_data)
min_var.solve()

weights = min_var.results.get('weights')

# Solve the optimization problem and print the weights
print(weights)

[ 0.03172767  0.05512308  0.03419218  0.10340288  0.04267807 -0.04278319
  0.00148504 -0.03616112 -0.00814935  0.01094959  0.0698288   0.01329095
 -0.04765833 -0.00598719 -0.00618123  0.03575352  0.01945158 -0.04292947
  0.03910342 -0.04072196  0.00928546  0.02859557 -0.00478851 -0.00988215
  0.01664949  0.03270035  0.00689021  0.04858716  0.02682633  0.08481336
 -0.00218284  0.02851922  0.04050138 -0.00262737  0.01547366  0.06256683
  0.03125978  0.01748976  0.02672408  0.01558699  0.00694583  0.00280747
  0.03492268  0.05181215  0.07323405 -0.01357452 -0.05114395  0.08471078
  0.03181276  0.07906905]


/Users/oscar/Documents/QPWP/qpmwp-course/src/optimization/constraints.py:49: UserWarning: Existing budget constraint is overwritten

  warnings.warn("Existing budget constraint is overwritten\n")
